# PY-09 | Spatial Queries & Spatial Joins με GeoPandas

Στα PY-07 και PY-08 δουλέψαμε με attributes, geometry και CRS. Τώρα περνάμε σε μία από τις βασικές ιδέες του GIS: **να επιλέγουμε και να συνδέουμε features με βάση τη χωρική τους σχέση**.

Σε αυτό το μάθημα χρησιμοποιούμε **απευθείας το boundary ZIP των 333 Δήμων της ΕΛΣΤΑΤ (`DHMOI_2021.zip`)**, χωρίς να χρειαζόμαστε το GeoPackage που δημιουργήθηκε στο PY-07. Μέσα στο notebook θα δημιουργήσουμε επίσης ένα μικρό, αναπαραγώγιμο dataset από συνθετικές μονάδες υγείας.

> **Κεντρική ιδέα:** στο attribute join ρωτάμε «ποιο ID ταιριάζει;». Στο spatial join ρωτάμε «ποια γεωμετρία σχετίζεται χωρικά με ποια άλλη;».


## 1. Στόχοι του μαθήματος

Στο τέλος του PY-09 θα μπορούμε να:

- εξηγούμε τι είναι ένα **spatial predicate**,
- χρησιμοποιούμε `within()`, `contains()` και `intersects()`,
- δημιουργούμε Boolean spatial queries,
- ελέγχουμε ότι δύο layers έχουν συμβατό CRS,
- χρησιμοποιούμε `gpd.sjoin()` για spatial joins,
- εξηγούμε πώς το `how=` και το `predicate=` αλλάζουν το αποτέλεσμα,
- εντοπίζουμε unmatched features με το `index_right`,
- καταλαβαίνουμε γιατί η κατεύθυνση ενός spatial join έχει σημασία,
- αναγνωρίζουμε one-to-many αποτελέσματα,
- υπολογίζουμε πόσες μονάδες βρίσκονται σε κάθε Δήμο.


## 2. Imports και paths

Χρησιμοποιούμε το **αρχικό, μη τροποποιημένο** boundary ZIP της ΕΛΣΤΑΤ με τους Δήμους του 2021. Το αρχείο διανέμεται μαζί με το course repository στον φάκελο `Data/course/vector/`.

Αναμενόμενη δομή project:

```text
Data/
├── course/
│   └── vector/
│       └── DHMOI_2021.zip
└── processed/
    └── DHMOI_2021/        # δημιουργείται τοπικά από το notebook
```

Το repository περιλαμβάνει μόνο το **αρχικό `DHMOI_2021.zip` χωρίς αλλαγές**. Το notebook κάνει extraction σε `Data/processed/`, ώστε τυχόν working/derived αρχεία να παραμένουν τοπικά και να μην αναδιανέμονται από το repository.

Οι μονάδες υγείας δημιουργούνται συνθετικά μέσα στο notebook, ώστε όλοι να παίρνουν τα ίδια point δεδομένα.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

from IPython.display import display

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point

BOUNDARY_ZIP = Path("Data/course/vector/DHMOI_2021.zip")
BOUNDARY_DIR = Path("Data/processed/DHMOI_2021")


## 3. Προετοιμασία και φόρτωση των Δήμων

Ένα shapefile αποτελείται από πολλά αρχεία που λειτουργούν μαζί (`.shp`, `.dbf`, `.shx`, `.prj` κ.ά.). Γνωρίζουμε ότι το layer που χρειαζόμαστε μέσα στο ZIP ονομάζεται **`Municipalities2021.shp`**.

Εξάγουμε το archive μόνο όταν χρειάζεται και αναζητούμε το shapefile αναδρομικά, ώστε ο κώδικας να λειτουργεί ακόμη κι αν το ZIP δημιουργήσει έναν επιπλέον υποφάκελο.


In [ ]:
if not BOUNDARY_ZIP.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το boundary ZIP: {BOUNDARY_ZIP}"
    )

shapefile_matches = list(
    BOUNDARY_DIR.rglob("Municipalities2021.shp")
)

if not shapefile_matches:
    BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

    with ZipFile(BOUNDARY_ZIP) as archive:
        archive.extractall(BOUNDARY_DIR)

    shapefile_matches = list(
        BOUNDARY_DIR.rglob("Municipalities2021.shp")
    )

if not shapefile_matches:
    raise FileNotFoundError(
        f"Δεν βρέθηκε το Municipalities2021.shp μέσα στο {BOUNDARY_DIR}"
    )

SHAPEFILE_PATH = shapefile_matches[0]

municipalities = gpd.read_file(SHAPEFILE_PATH)

required_columns = {
    "CODE",
    "NAME_GR",
    "geometry",
}
missing_columns = required_columns - set(municipalities.columns)

assert not missing_columns, (
    f"Λείπουν στήλες από το boundary layer: {missing_columns}"
)
assert len(municipalities) == 333
assert municipalities.crs.to_epsg() == 2100

print("Shapefile:", SHAPEFILE_PATH)
print("Shape:", municipalities.shape)
print("CRS:", municipalities.crs)
municipalities.head()


## 4. Δημιουργία συνθετικών μονάδων υγείας

Δεν χρειαζόμαστε δεύτερο download. Θα δημιουργήσουμε **120 σημεία μέσα σε Δήμους** με σταθερό random seed.

Θα προσθέσουμε επίσης δύο ειδικά σημεία:

- ένα πάνω σε boundary Δήμου, επιλεγμένο έτσι ώστε να **μη βρίσκεται στο εσωτερικό κανενός Δήμου**,
- ένα έξω από την έκταση της Ελλάδας.

Αυτά τα δύο σημεία θα μας βοηθήσουν να δούμε καθαρά τη διαφορά ανάμεσα στα spatial predicates.


In [ ]:
rng = np.random.default_rng(42)

def random_point_inside(geometry):
    minx, miny, maxx, maxy = geometry.bounds

    for _ in range(10_000):
        candidate = Point(
            rng.uniform(minx, maxx),
            rng.uniform(miny, maxy),
        )

        if geometry.contains(candidate):
            return candidate

    return geometry.representative_point()


In [ ]:
selected_indices = rng.choice(
    municipalities.index.to_numpy(),
    size=120,
    replace=True,
)

facility_types = np.array([
    "Κέντρο Υγείας",
    "Περιφερειακό Ιατρείο",
    "Κλινική",
])

records = []

for i, municipality_index in enumerate(selected_indices, start=1):
    municipality_geometry = municipalities.loc[
        municipality_index,
        "geometry",
    ]

    records.append({
        "facility_id": f"F{i:03d}",
        "facility_type": rng.choice(facility_types),
        "capacity": int(rng.integers(10, 151)),
        "geometry": random_point_inside(municipality_geometry),
    })


In [ ]:
def find_boundary_demo_point(polygons):
    fractions = (0.1, 0.25, 0.5, 0.75, 0.9)

    for geometry in polygons.geometry:
        for fraction in fractions:
            candidate = geometry.boundary.interpolate(
                fraction,
                normalized=True,
            )

            if (
                polygons.intersects(candidate).any()
                and not polygons.contains(candidate).any()
            ):
                return candidate

    raise RuntimeError(
        "Δεν βρέθηκε κατάλληλο boundary point για το demo."
    )

boundary_point = find_boundary_demo_point(municipalities)

minx, miny, maxx, maxy = municipalities.total_bounds
outside_point = Point(
    maxx + 100_000,
    maxy + 100_000,
)

records.extend([
    {
        "facility_id": "F121",
        "facility_type": "Boundary demo",
        "capacity": 25,
        "geometry": boundary_point,
    },
    {
        "facility_id": "F122",
        "facility_type": "Outside demo",
        "capacity": 25,
        "geometry": outside_point,
    },
])

facilities = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs=municipalities.crs,
)

print("Facilities:", len(facilities))
facilities.tail()


## 5. Επιθεώρηση του point layer

Ένα `GeoDataFrame` μπορεί να περιέχει points, lines ή polygons. Εδώ η ενεργή geometry column περιέχει αντικείμενα `Point`.


In [ ]:
print("Type:", type(facilities))
print("Geometry types:")
print(facilities.geom_type.value_counts())

print()
print("CRS:", facilities.crs)

facilities.head()


## 6. Πρώτος χάρτης: points πάνω σε polygons

Σχεδιάζουμε πρώτα τους Δήμους και μετά τις μονάδες υγείας πάνω στον ίδιο `Axes`.


In [ ]:
ax = municipalities.plot(
    figsize=(9, 9),
    facecolor="none",
    edgecolor="0.65",
    linewidth=0.35,
)

facilities.plot(
    ax=ax,
    markersize=18,
)

ax.set_title("Synthetic health facilities and municipalities")
ax.set_axis_off()
plt.show()


## 7. CRS compatibility πριν από spatial operation

Ένα spatial join συγκρίνει τις **πραγματικές coordinate values** των δύο layers. Το ότι δύο layers «φαίνονται σωστά» σε ένα GIS δεν αρκεί.

Πριν από spatial query ή spatial join, ελέγχουμε το CRS.


In [ ]:
print("Municipalities CRS:", municipalities.crs)
print("Facilities CRS:", facilities.crs)
print("Same CRS:", municipalities.crs == facilities.crs)

assert municipalities.crs == facilities.crs


## 8. Spatial predicates: η σχέση μεταξύ δύο γεωμετριών

Ένα **spatial predicate** επιστρέφει `True` ή `False` για μία χωρική σχέση.

Στο μάθημα θα χρησιμοποιήσουμε κυρίως:

- `point.within(polygon)` → το point βρίσκεται στο εσωτερικό του polygon,
- `polygon.contains(point)` → το polygon περιέχει το point,
- `point.intersects(polygon)` → οι δύο geometries έχουν τουλάχιστον ένα κοινό σημείο.

Για ένα point αυστηρά μέσα σε polygon:

```text
point.within(polygon) == True
polygon.contains(point) == True
```

Στα όρια όμως η συμπεριφορά αλλάζει.


## 9. `within`, `contains` και το όριο ενός polygon

Το ειδικό `F121` έχει επιλεγεί πάνω σε boundary Δήμου και έχουμε ελέγξει ότι **δεν βρίσκεται στο εσωτερικό κανενός Δήμου**. Έτσι έχουμε ένα καθαρό edge case για να συγκρίνουμε `within`, `contains` και `intersects`.


In [ ]:
boundary_point = facilities.loc[
    facilities["facility_id"] == "F121",
    "geometry",
].iloc[0]

boundary_municipalities = municipalities.loc[
    municipalities.intersects(boundary_point)
]

assert not boundary_municipalities.empty

demo_polygon = boundary_municipalities.geometry.iloc[0]
inside_point = demo_polygon.representative_point()

print("Inside point:")
print("  within:", inside_point.within(demo_polygon))
print("  intersects:", inside_point.intersects(demo_polygon))
print("  polygon contains point:", demo_polygon.contains(inside_point))

print()
print("Boundary point:")
print("  within:", boundary_point.within(demo_polygon))
print("  intersects:", boundary_point.intersects(demo_polygon))
print("  polygon contains point:", demo_polygon.contains(boundary_point))

print()
print(
    "F121 within any municipality:",
    municipalities.contains(boundary_point).any(),
)


Το σημαντικό σημείο είναι ότι **`within` και `contains` απαιτούν το point να βρίσκεται στο εσωτερικό**. Ένα boundary point δεν είναι `within`, αλλά εξακολουθεί να `intersects` το polygon.

Άρα το predicate δεν είναι τεχνική λεπτομέρεια: εκφράζει **τι ακριβώς εννοούμε** με τη χωρική σχέση.


## 10. Boolean spatial query

Μπορούμε να χρησιμοποιήσουμε ένα spatial predicate όπως χρησιμοποιούμε ένα Boolean mask στην Pandas.

Θα επιλέξουμε έναν Δήμο που γνωρίζουμε ότι έχει τουλάχιστον ένα από τα συνθετικά σημεία.


In [ ]:
query_municipality_index = selected_indices[0]

query_municipality = municipalities.loc[
    [query_municipality_index]
]

query_geometry = query_municipality.geometry.iloc[0]

within_mask = facilities.within(query_geometry)

facilities_in_query_municipality = facilities.loc[
    within_mask
]

print(
    "Municipality:",
    query_municipality["NAME_GR"].iloc[0],
)
print(
    "Facilities strictly within:",
    len(facilities_in_query_municipality),
)

facilities_in_query_municipality


## 11. Spatial join με `gpd.sjoin()`

Το Boolean query απαντά μία ερώτηση για **ένα συγκεκριμένο polygon**. Το spatial join μπορεί να κάνει το ίδιο για **όλα τα features ταυτόχρονα**.

Στόχος:

```text
facility point
      +
municipality polygon
      ↓
ποιος Δήμος περιέχει κάθε facility;
```

Στο `gpd.sjoin()`:

- το πρώτο GeoDataFrame είναι το **left** dataset,
- το δεύτερο GeoDataFrame είναι το **right** dataset,
- `how="left"` κρατά όλα τα rows του left dataset,
- `predicate="within"` ορίζει τη χωρική σχέση που πρέπει να ισχύει.


In [ ]:
municipality_lookup = municipalities[
    [
        "CODE",
        "NAME_GR",
        "geometry",
    ]
].copy()

joined_within = gpd.sjoin(
    facilities,
    municipality_lookup,
    how="left",
    predicate="within",
)

joined_within[
    [
        "facility_id",
        "facility_type",
        "NAME_GR",
        "index_right",
    ]
].head(10)


## 12. Τι είναι το `index_right`;

Το GeoPandas προσθέτει το `index_right` για να μας δείξει **ποιο row του right GeoDataFrame έκανε match**.

Αν δεν βρεθεί spatial match, το `index_right` είναι missing (`NaN`).

Αυτό είναι πολύ χρήσιμο για validation.


In [ ]:
unmatched_within = joined_within.loc[
    joined_within["index_right"].isna()
]

print("Unmatched with predicate='within':")
display(
    unmatched_within[
        [
            "facility_id",
            "facility_type",
        ]
    ]
)

assert set(unmatched_within["facility_id"]) == {"F121", "F122"}


Περιμένουμε να δούμε τα δύο ειδικά σημεία:

- `F121` — βρίσκεται σε boundary αλλά όχι στο εσωτερικό κανενός Δήμου, άρα δεν είναι `within`,
- `F122` — βρίσκεται έξω από την Ελλάδα.

Τα 120 τυχαία σημεία δημιουργήθηκαν με `geometry.contains(candidate)`, άρα βρίσκονται στο εσωτερικό ενός Δήμου.


## 13. Αλλάζοντας predicate: `intersects`

Τώρα επαναλαμβάνουμε το spatial join με `predicate="intersects"`.

Το `F121` θα μπορεί πλέον να κάνει match, επειδή ένα point πάνω στο boundary **intersects** το polygon.


In [ ]:
joined_intersects = gpd.sjoin(
    facilities,
    municipality_lookup,
    how="left",
    predicate="intersects",
)

boundary_matches = joined_intersects.loc[
    joined_intersects["facility_id"] == "F121",
    [
        "facility_id",
        "facility_type",
        "NAME_GR",
        "index_right",
    ],
]

boundary_matches


In [ ]:
unmatched_intersects = joined_intersects.loc[
    joined_intersects["index_right"].isna()
]

print("Unmatched with predicate='intersects':")
display(
    unmatched_intersects[
        [
            "facility_id",
            "facility_type",
        ]
    ]
)

assert set(unmatched_intersects["facility_id"]) == {"F122"}


Ένα boundary point μπορεί να intersect περισσότερα από ένα polygons όταν βρίσκεται πάνω σε κοινό όριο. Αυτό είναι ένα παράδειγμα όπου ένα spatial join μπορεί να παράγει περισσότερα rows από όσα είχε το αρχικό point layer.


## 14. Η κατεύθυνση του spatial join έχει σημασία

Μέχρι τώρα είχαμε:

```text
LEFT:  facilities
RIGHT: municipalities
```

και ρωτούσαμε:

> «Σε ποιον Δήμο βρίσκεται κάθε facility;»

Μπορούμε όμως να αντιστρέψουμε την ερώτηση:

```text
LEFT:  municipalities
RIGHT: facilities
```

και να ρωτήσουμε:

> «Ποια facilities περιέχει κάθε Δήμος;»


In [ ]:
municipalities_left = gpd.sjoin(
    municipality_lookup,
    facilities[
        [
            "facility_id",
            "facility_type",
            "capacity",
            "geometry",
        ]
    ],
    how="left",
    predicate="contains",
)

municipalities_left[
    [
        "CODE",
        "NAME_GR",
        "facility_id",
        "facility_type",
    ]
].head(10)


## 15. One-to-many αποτέλεσμα

Ένας Δήμος μπορεί να περιέχει **πολλά** facilities. Σε αυτή την περίπτωση το spatial join επαναλαμβάνει το row του Δήμου μία φορά για κάθε matching point.

Αυτό δεν είναι λάθος ή duplication bug. Είναι η φυσική αναπαράσταση μιας **one-to-many** σχέσης.


In [ ]:
facility_rows_per_municipality = (
    municipalities_left
    .dropna(subset=["facility_id"])
    .groupby(
        [
            "CODE",
            "NAME_GR",
        ]
    )
    .size()
    .sort_values(ascending=False)
)

facility_rows_per_municipality.head(10)


## 16. Μέτρηση facilities ανά Δήμο

Για χαρτογράφηση θέλουμε πάλι **ένα row ανά Δήμο**. Θα ομαδοποιήσουμε λοιπόν τα successful matches του `joined_within` με βάση το `index_right`.


In [ ]:
facility_counts = (
    joined_within
    .dropna(subset=["index_right"])
    .groupby("index_right")
    .size()
    .rename("facility_count")
)

municipality_facilities = municipalities.join(
    facility_counts,
    how="left",
)

municipality_facilities["facility_count"] = (
    municipality_facilities["facility_count"]
    .fillna(0)
    .astype(int)
)

municipality_facilities[
    [
        "CODE",
        "NAME_GR",
        "facility_count",
    ]
].head()


In [ ]:
print(
    "Facilities counted inside municipalities:",
    municipality_facilities["facility_count"].sum(),
)
print(
    "Municipalities with zero facilities:",
    (municipality_facilities["facility_count"] == 0).sum(),
)


Σε ένα καθαρό, μη επικαλυπτόμενο municipality layer, το άθροισμα του `facility_count` πρέπει να είναι **120**: μετράμε μόνο τα 120 points που δημιουργήθηκαν αυστηρά μέσα σε polygons. Τα δύο demo points δεν μπαίνουν σε αυτό το count με `predicate="within"`.


## 17. Χαρτογράφηση του αριθμού facilities

Τώρα το αποτέλεσμα είναι πάλι ένα polygon GeoDataFrame, αλλά έχει αποκτήσει μία νέα μεταβλητή: `facility_count`.


In [ ]:
ax = municipality_facilities.plot(
    column="facility_count",
    figsize=(9, 9),
    legend=True,
    edgecolor="0.55",
    linewidth=0.25,
)

ax.set_title("Synthetic facilities per municipality")
ax.set_axis_off()
plt.show()


## 18. Καλή πρακτική για παρόχους spatial data

Για datasets που προορίζονται για spatial queries και joins, είναι ιδιαίτερα χρήσιμο να παρέχονται:

- σταθερά και μοναδικά feature IDs,
- έγκυρο CRS metadata,
- valid geometries,
- σαφής ορισμός του τι αντιπροσωπεύει κάθε geometry,
- machine-readable formats όπως GeoPackage ή GeoJSON,
- τεκμηρίωση για το αν points πάνω σε boundaries έχουν ειδική σημασία.

Το spatial join είναι τόσο αξιόπιστο όσο τα geometries και τα metadata που του δίνουμε.


## 19. Ασκήσεις

### Άσκηση 1

Πόσα facilities δεν έκαναν match με `predicate="within"`; Εξήγησε γιατί.

### Άσκηση 2

Για τον Δήμο `query_municipality`, σύγκρινε:

```python
facilities.within(query_geometry)
```

με:

```python
facilities.intersects(query_geometry)
```

Έχουν πάντα το ίδιο αποτέλεσμα; Γιατί;

### Άσκηση 3

Εντόπισε το `F121` στο `joined_intersects`. Με πόσα polygons έκανε match;

### Άσκηση 4

Υπολόγισε πόσοι Δήμοι έχουν `facility_count == 0`.

### Άσκηση 5

Εμφάνισε τους 10 Δήμους με το μεγαλύτερο `facility_count`.


In [ ]:
# Άσκηση 1


In [ ]:
# Άσκηση 2


In [ ]:
# Άσκηση 3


In [ ]:
# Άσκηση 4


In [ ]:
# Άσκηση 5


## 20. Σύνοψη

Στο PY-09 περάσαμε από την attribute λογική στη spatial λογική:

```text
geometry + CRS
      ↓
spatial predicate
      ↓
Boolean spatial query
      ↓
gpd.sjoin()
      ↓
inspect unmatched / one-to-many
      ↓
aggregate spatial matches
      ↓
map the result
```

Οι βασικές ιδέες είναι:

1. **Το predicate εκφράζει τη γεωγραφική ερώτηση.** `within`, `contains` και `intersects` δεν είναι εναλλάξιμα.
2. **Το CRS πρέπει να είναι συμβατό πριν από spatial operations.**
3. **Το `how=` αφορά το ποια rows διατηρούμε, ενώ το `predicate=` αφορά τη χωρική σχέση.**
4. **Το `index_right` είναι σημαντικό validation εργαλείο.**
5. **One-to-many rows είναι αναμενόμενα όταν ένα feature σχετίζεται με πολλά features του άλλου layer.**

Στο PY-10 θα χρησιμοποιήσουμε αυτή τη βάση για **vector geoprocessing**: buffer, clip, dissolve και overlay.


## 21. Λύσεις ασκήσεων

Προσπάθησε πρώτα να λύσεις τις ασκήσεις του **Τμήματος 19** χωρίς να κοιτάξεις παρακάτω.


### Άσκηση 1 — Unmatched facilities


In [ ]:
exercise_unmatched = joined_within.loc[
    joined_within["index_right"].isna(),
    [
        "facility_id",
        "facility_type",
    ],
]

print("Number unmatched:", len(exercise_unmatched))
exercise_unmatched


Με `predicate="within"` περιμένουμε δύο unmatched demo points: το boundary point και το outside point.


### Άσκηση 2 — `within` vs `intersects`


In [ ]:
exercise_within = facilities.within(
    query_geometry
)
exercise_intersects = facilities.intersects(
    query_geometry
)

print("Within:", exercise_within.sum())
print("Intersects:", exercise_intersects.sum())
print(
    "Same Boolean mask:",
    exercise_within.equals(exercise_intersects),
)


Δεν είναι εγγυημένο ότι τα δύο masks είναι ίδια. Ένα point πάνω στο boundary μπορεί να `intersects` το polygon χωρίς να είναι `within` του polygon.


### Άσκηση 3 — Matches του `F121`


In [ ]:
f121_matches = joined_intersects.loc[
    joined_intersects["facility_id"] == "F121"
]

print("F121 matches:", len(f121_matches))

f121_matches[
    [
        "facility_id",
        "NAME_GR",
        "index_right",
    ]
]


### Άσκηση 4 — Δήμοι χωρίς facilities


In [ ]:
zero_facility_municipalities = (
    municipality_facilities["facility_count"] == 0
).sum()

print(
    "Municipalities with zero facilities:",
    zero_facility_municipalities,
)


### Άσκηση 5 — Top 10


In [ ]:
municipality_facilities.nlargest(
    10,
    "facility_count",
)[
    [
        "CODE",
        "NAME_GR",
        "facility_count",
    ]
]
